# 02 — Kiểm chứng chỉ số M1–M8, M12–M13

**CHỈ CHẠY VỚI FIXTURE SYNTHETIC. KHÔNG PHẢI KẾT QUẢ ĐỒ ÁN.**

Các công thức nằm trong `src/analysis_core/metrics.py`; xem `docs/metric_contract.md` để biết mẫu số, oracle, hành vi N/A và giới hạn. M2 phải tách 3 chặng. M9–M11 chuyển sang Notebook 03.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Notebook must run from this repository folder or one of its subfolders")
SYNTHETIC = ROOT / "data" / "sample_synthetic"
print("Repository:", ROOT)
from analysis_core.metrics import (audit_summary, m2_field_accuracy, m8_fhir_validity)

def get(file):
    return pd.read_csv(SYNTHETIC / file)

cases = get("cases.csv")
comparisons = get("comparisons.csv")
targets = get("target_records.csv")
transactions = get("transactions.csv")
error_cases = get("error_cases.csv")
validations = get("fhir_validation.csv")
recovery = get("recovery_cases.csv")
requirements = get("requirements.csv")
print("⚠️ ALL INPUTS ARE SYNTHETIC, WITH INTENTIONAL FAILURES")


## A. Chỉ số tổng quan

Bộ khung tính các tỷ lệ **trên bảng đã chuẩn hóa và phải có oracle độc lập**. Nó chưa đọc được `source.jsonl/fhir.jsonl/target.jsonl` của prototype. `numerator`, `denominator`, `percentage` minh bạch; số đếm M4–M6 không có mẫu số.

In [ ]:
report = audit_summary(cases, comparisons, targets, transactions, error_cases, validations, recovery, requirements)
display(report)


## B. M2: Tỷ lệ đúng ở ba chặng

Không gộp ba chặng thành một con số chung khi chưa có định nghĩa. Missing comparison trong tập A phải được ghi thành FALSE, không được xóa dòng.

In [ ]:
display(m2_field_accuracy(comparisons))


## C. M8: Đếm cả kết quả hợp lệ không sinh Observation

M8 có mẫu số là Observation đã sinh, vì vậy phải ghi số nguồn hợp lệ không tạo resource để tránh báo cáo lạc quan.

In [ ]:
m8, not_generated = m8_fhir_validity(validations)
print("M8:", m8.as_dict())
print("Valid inputs without generated Observation:", not_generated)


## D. Đối chiếu với báo cáo thực tế (chưa thực hiện)

Khi có evidence chính thức, tạo adapter có truy vết từ `run_id/case_id/exchange_id` đến `source.jsonl`, `target.jsonl`, `comparisons.csv` và trạng thái lịch sử. Chạy lại `pytest` sau khi xác định expected output độc lập. Không dùng dữ liệu synthetic để thay chỗ dữ liệu còn thiếu.